## 中文语音自动识别

### 1. 模型说明

- [Paraformer语音识别-中文-通用-16k-离线-large-pytorch](https://modelscope.cn/models/iic/speech_paraformer-large_asr_nat-zh-cn-16k-common-vocab8404-pytorch)
  > - 热词版本：：[Paraformer-large热词版模型](https://www.modelscope.cn/models/iic/speech_paraformer-large-contextual_asr_nat-zh-cn-16k-common-vocab8404/summary)支持热词定制功能，基于提供的热词列表进行激励增强，提升热词的召回率和准确率。
  > - 长音频版本：[Paraformer-large长音频模型](https://www.modelscope.cn/models/iic/speech_paraformer-large-vad-punc_asr_nat-zh-cn-16k-common-vocab8404-pytorch/summary),集成VAD, ASR,标点与时间戳功能，可直接对时长为数小时音频进行识别，并输出带标点文字与时间戳。
- [FSMN语音端点检测-中文-通用-16k](https://modelscope.cn/models/iic/speech_fsmn_vad_zh-cn-16k-common-pytorch)
  > 16K中文通用VAD模型：可用于检测长语音片段中有效语音的起止时间点。
  > - 基于[Paraformer-large长音频模型](https://www.modelscope.cn/models/iic/speech_paraformer-large-vad-punc_asr_nat-zh-cn-16k-common-vocab8404-pytorch/summary)场景的使用
  > - 基于FunASR框架，可进行ASR、VAD，中文标点的自由组合
  > - **基于音频数据的有效语音片段起止时间点检测**
- [CT-Transformer标点-中文-通用-pytorch](https://modelscope.cn/models/iic/punc_ct-transformer_zh-cn-common-vocab272727-pytorch)
  > 中文标点通用模型：可用于语音识别模型输出文本的标点预测。
  > - 基于[Paraformer-large长音频模型](https://www.modelscope.cn/models/iic/speech_paraformer-large-vad-punc_asr_nat-zh-cn-16k-common-vocab8404-pytorch/summary)场景的使用
  > - 基于FunASR框架，可进行ASR、VAD，标点的自由组合
  > - **基于纯文本输入的标点预测**

**我是提前把模型下载到了本地文件中：**

In [1]:
# 模型存放的根目录
model_root_dir = "../../../models/modelscope"

In [2]:
!ls {model_root_dir} | grep "punc_ct\|speech_fsmn_vad\|speech_paraformer-large_asr"

punc_ct-transformer_zh-cn-common-vocab272727-pytorch
speech_fsmn_vad_zh-cn-16k-common-pytorch
speech_paraformer-large_asr_nat-zh-cn-16k-common-vocab8404-pytorch


### 2. 使用funasr识别中文语音

In [3]:
import os
import traceback
from tqdm import tqdm

from funasr import AutoModel

#### 2.1 准备模型路径

In [4]:
# 语音识别模型
asr_model_path = "{}/speech_paraformer-large_asr_nat-zh-cn-16k-common-vocab8404-pytorch".format(model_root_dir)
# 端点检测模型
vad_model_path = "{}/speech_fsmn_vad_zh-cn-16k-common-pytorch".format(model_root_dir)
# 预测标点模型punc
punc_model_path = "{}/punc_ct-transformer_zh-cn-common-vocab272727-pytorch".format(model_root_dir)

In [5]:
# 如果以上3个模型不存在，那么我们就自动下载模型
asr_model_path  = asr_model_path  if os.path.exists(asr_model_path) else "iic/speech_paraformer-large_asr_nat-zh-cn-16k-common-vocab8404-pytorch"
vad_model_path  = vad_model_path  if os.path.exists(vad_model_path) else "iic/speech_fsmn_vad_zh-cn-16k-common-pytorch"
punc_model_path = punc_model_path if os.path.exists(punc_model_path) else "iic/punc_ct-transformer_zh-cn-common-vocab272727-pytorch"

# 如果是在线下载模型，可以传递模型仓库：model_hub：ms是modelscope,hf是huggingface下载模型

#### 2.2 实例化model

In [16]:
model = AutoModel(
    model              = asr_model_path,
    model_verion       = "v2.0.4",
    # 如果太短的语音，进行端点检测和预测标点会报错的
    vad_model          = vad_model_path,
    vad_model_version  = "v2.0.4",
    punc_model         = punc_model_path,
    punc_model_version = "v2.0.4"
)

In [7]:
type(model)

funasr.auto.auto_model.AutoModel

#### 2.3 识别音频文本

In [8]:
# filepath = '../../../data/audio/moda/asr_example.wav'
# filepath = '../../../data/audio/moda/vad_example.wav'
# filepath = '../../../data/audio/alex/001.m4a'
filepath = '../../../data/audio/test/001.wav'
!ls {filepath}

../../../data/audio/test/001.wav


In [9]:
try:
    result = model.generate(input=filepath)
except Exception as e:
    print("出现错误：{}".format(str(e)))
    print(traceback.format_exc())
    
# type(result): 结果是个数组

100%|█████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  3.55it/s]
{'load_data': '0.000', 'extract_feat': '0.006', 'forward': '0.282', 'batch_size': '1', 'rtf': '0.034'}, : 10
rtf_avg: 0.034: 100%|█████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  3.52it/s]

  0%|                                                                                 | 0/1 [00:00<?, ?it/s]
{'load_data': 0.0, 'extract_feat': 0.0, 'forward': '0.022', 'batch_size': '1', 'rtf': '-0.022'}, : 100%|█| 1
rtf_avg: -0.022: 100%|████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 42.62it/s]
rtf_avg: 0.038, time_speech:  8.211, time_escape: 0.312: 100%|████████████████| 1/1 [00:00<00:00,  3.18it/s]


In [10]:
result

[{'key': '001',
  'text': '足够好，谢谢。好，非常感谢那个三宝老师的回答。三宝老师说，我们在整个店铺的这个活动当中，我们要学会换位思考。其实。'}]

In [17]:
# model.generate(input='https://isv-data.oss-cn-hangzhou.aliyuncs.com/ics/MaaS/ASR/test_audio/asr_example_zh.wav')